# Building Profiles with CIMTool

## What is CIMTool?

[CIMTool](https://cimtool.ucaiug.io/) is an open source tool maintained by the UCAIug CIM Users Group community for working with the CIM canonical model to produce design artifacts such as database schemas, message exchange syntax, source code classes, and reference documentation. Releases are published on [GitHub](https://github.com/cimug-org/CIMTool/releases).

CIMTool operates on **Contextual Profiles** - UML-based subsets of the full CIM that define which classes, attributes, and associations are needed for a specific application. These Contextual Profiles are then used by "builders" (XSLT transforms) to generate various output formats.

## CIMTool Capabilities

CIMTool provides comprehensive functionality for profile management:

### Profile Creation and Management
* Create Contextual Profiles from the CIM Unified Modeling Language (UML) model
* Import Contextual Profiles from spreadsheets
* Constrain cardinality of attributes
* Remove unwanted classes and attributes
* Document profile-specific usage notes

### Schema Generation
* Generate Resource Description Framework Schema (RDF Schema) from Contextual Profiles
* Create XML Schema Definition (XSD) files
* Produce JSON schemas
* Generate custom formats using XSLT builders

### Validation
* Validate Contextual Profile schemas against CIM UML
* Validate data instances against a Contextual Profile schema
* Validate incremental data instances against an existing instance

### Extensibility
* Create custom builders using XSLT transforms
* Import custom builders to generate specialized outputs:
  - Word documentation
  - SQL database scripts
  - Python dataclass schemas (CIMantic Graphs)
  - Custom application code

## The CIMantic Graphs Custom Builder

While CIMTool provides several built-in builders (RDF Schema, XSD, etc.), CIMantic Graphs uses a **custom XSLT builder** to generate Python-specific output. It is maintained in the [`CIMTool_plugin`](https://github.com/PNNL-CIM-Tools/CIM-Profiles/tree/main/CIMTool_plugin) folder of the CIM-Profiles repository.

The CIMantic Graphs builder consists of **two companion builders**:

1. **cimantic-graphs.xsl** - Generates the main Python dataclass schema file
2. **cimantic-graphs-init.xsl** - Generates the `__init__.py` file for proper Python imports

These builders transform a CIMTool Contextual Profile into a hierarchical tree of Python dataclasses that serve as the "single-source-of-truth" for:
- Database query generation
- Graph traversal algorithms
- Type checking and validation
- IDE autocomplete and introspection

## Generated Python Dataclass Structure

The CIMantic Graphs builder produces Python dataclasses with rich metadata that preserves the semantic information from the CIM UML model.

### Key Features of Generated Dataclasses:

1. **Inheritance Hierarchy** - Preserves CIM class inheritance (e.g., `Terminal` → `ACDCTerminal` → `IdentifiedObject` → `Identity`)
2. **Type Annotations** - Full Python type hints for all attributes
3. **Field Metadata** - Embedded UML metadata including:
   - Field type (`Attribute`, `Association`, `enumeration`)
   - Cardinality (`minOccurs`, `maxOccurs`)
   - Inverse relationships for bidirectional associations
   - The attribute's `namespace`, and `serialize`: which end of an association is written to files
4. **Docstrings** - CIM documentation embedded in Python docstrings

### Example Generated Code:

An excerpt of the generated `cimprofiles.cim17.cimhub_2023` profile (docstrings shortened):

```python
from cimprofiles.identity import Identity, stereotype
from cimprofiles.units import CIMUnit, UnitMultiplier, UnitSymbol

@dataclass(repr=False, eq=False)
class IdentifiedObject(Identity):
    '''
    This is a root class to provide common identification for all classes needing
    identification and naming attributes.
    '''

    mRID: Optional[str] = field(
        default=None,
        metadata={
        'type': 'Attribute',
        'stereotypes': ['Attribute'],
        'minOccurs': '0',
        'maxOccurs': '1',
        'namespace': 'http://iec.ch/TC57/CIM100#',
        'serialize': True,
        })
    '''
    Master resource identifier issued by a model authority. ...
    '''

@dataclass(repr=False, eq=False)
class Terminal(ACDCTerminal):
    '''
    An AC electrical connection point to a piece of conducting equipment. Terminals
    are connected at physical connection points called connectivity nodes.
    '''

    ConnectivityNode: Optional[ConnectivityNode] = field(
        default=None,
        metadata={
        'type': 'Association',
        'stereotypes': [],
        'minOccurs': '0',
        'maxOccurs': '1',
        'inverse': 'ConnectivityNode.Terminals',
        'namespace': 'http://iec.ch/TC57/CIM100#',
        'serialize': True,
        })
    '''
    The connectivity node to which this terminal connects with zero impedance.
    '''
```

The same metadata can be read from the installed profile:

In [1]:
import cimprofiles.cim17.cimhub_2023 as cim

print(' -> '.join(cls.__name__ for cls in cim.Terminal.__mro__[:-1]))
dict(cim.Terminal.__dataclass_fields__['ConnectivityNode'].metadata)

Terminal -> ACDCTerminal -> IdentifiedObject -> Identity


{'type': 'Association',
 'stereotypes': [],
 'minOccurs': '0',
 'maxOccurs': '1',
 'inverse': 'ConnectivityNode.Terminals',
 'namespace': 'http://iec.ch/TC57/CIM100#',
 'serialize': True}

## Step-by-Step: Building a Profile with CIMTool

### Prerequisites

1. **Install CIMTool** - Download a release from [GitHub](https://github.com/cimug-org/CIMTool/releases) (see the [getting started guide](https://cimtool.ucaiug.io/getting-started/))
2. **Obtain the CIM UML Model** - The base CIM UML (e.g., IEC 61970 CIM17v40) from the CIM Users Group
3. **Get the CIMantic Graphs Builders** - `cimantic-graphs.xsl` and `cimantic-graphs-init.xsl` from [`CIMTool_plugin`](https://github.com/PNNL-CIM-Tools/CIM-Profiles/tree/main/CIMTool_plugin)

### Step 1: Create or Open a Contextual Profile

In CIMTool:
- Create a new profile or open an existing `.owl` profile file
- Base it on the appropriate CIM UML version (e.g., CIM17, CIM18)
- Name your profile descriptively (e.g., "DistributionTopology", "TransmissionPlanning")

### Step 2: Select Classes and Attributes

Browse the CIM UML model and select:
- **Required classes** - Only classes needed for your use case
- **Required attributes** - Remove unnecessary attributes from selected classes
- **Associations** - Keep only relevant relationships between classes

**Best Practice:** Start with the minimum classes needed. You can always expand later.

### Step 3: Configure Cardinality

For each attribute and association:
- Review default cardinality from base CIM
- Constrain if needed (e.g., make optional attributes required)
- Document any constraints in profile notes

### Step 4: Add Profile Documentation

Document your profile with:
- Purpose and use case description
- Intended audience and applications
- Any deviations or constraints from base CIM
- Usage examples and patterns

### Step 5: Configure the CIMantic Graphs Builders

#### Import the Builders

1. In CIMTool, go to **"Maintain XSLT Transform Builders"**
2. Import **cimantic-graphs.xsl**:
   - Set type: `TEXT`
   - Set extension: `py`
3. Import **cimantic-graphs-init.xsl**:
   - Set type: `TEXT`
   - Set extension: `__init__.py`

**Important Note:** CIMTool requires unique file extensions for each builder. An artifact's name is the base name of the `.owl` profile followed by the builder's extension, so `__init__.py` gives the initialization file a distinct name.

#### Select the Builders

In the Profile Summary tab, select both CIMantic Graphs builders.

### Step 6: Generate the Data Profile

1. Click **"Save"** in CIMTool
2. CIMTool creates two files in the project's `Profiles` folder:
   - `<profile_name>.py` - Main dataclass schema
   - `<profile_name>.__init__.py` - Python package initialization

### Step 7: Install the Profile

A profile is an ordinary Python package. Keep it in your own package, or contribute it to `cimprofiles`.

#### Option A: Your Own Package

Copy the two generated files into a package that is importable in your environment, renaming `my_profile.__init__.py` to `__init__.py`:

```
my_package/
  __init__.py
  my_profile/
    __init__.py
    my_profile.py
```

The init builder imports from `cimgraph.data_profile.my_profile.my_profile`. Change that line in `__init__.py` to `from my_package.my_profile.my_profile import (`. Older builders also import `Identity` and the units from `cimgraph.data_profile`; point those at `cimprofiles.identity` and `cimprofiles.units`.

`python -m cimprofiles.dedupe_profile_fields` only processes the profiles inside `cimprofiles`. If your profile selects both ends of an association, check the generated file for fields defined twice (Python keeps the last definition).

#### Option B: Contribute It to `cimprofiles`

To share the profile upstream, add it to the [CIM-Profiles](https://github.com/PNNL-CIM-Tools/CIM-Profiles) repository, under the directory for its CIM version (`cim16`, `cim17`, `cim18`). In a checkout of that repository:

```bash
# Copy the generated files to src/cimprofiles/cim17/my_profile/ and point their
# imports at cimprofiles
scripts/import_cimtool_output.sh cim17 <path>/Profiles/my_profile.py <path>/Profiles/my_profile.__init__.py

# Remove duplicate field definitions emitted when a profile selects both ends of an association
uv run python -m cimprofiles.dedupe_profile_fields

# Only if the profile is part of a pre-baked merged profile listed in
# src/cimprofiles/merge_releases.yaml: rebuild the merged packages
uv run python -m cimprofiles.build_merged_profiles

uv run pytest
```

Adding the profile to `PROFILE_PACKAGES` in `src/cimprofiles/__init__.py` gives it a short name. A registered profile can also get a published JSON-LD context: add a units convention in `src/cimprofiles/jsonld/conventions/my_profile.yaml` and run `uv run python -m cimprofiles.jsonld.build my_profile`. The [CIM-Profiles README](https://github.com/PNNL-CIM-Tools/CIM-Profiles#updating-a-profile) has the full procedure. To use the checkout before it is released, install it into your cim-graph environment, e.g. `uv pip install -e ../CIM-Profiles`.

### Step 8: Use Your Custom Profile

Import your profile like the built-in ones (`import my_package.my_profile as cim`, or `import cimprofiles.cim17.my_profile as cim` for Option B), or select it for cim-graph's connections and models with `CIMG_CIM_PROFILE`. The variable takes any importable module path (`'my_package.my_profile'`, `'cimprofiles.cim17.my_profile'`) or a short name listed in `cimprofiles.PROFILE_PACKAGES`.

The example below uses the built-in `cimhub_2023` profile the same way:

In [2]:
import os

from cimgraph.core import get_cim_profile

os.environ['CIMG_CIM_PROFILE'] = 'cimprofiles.cim17.cimhub_2023'  # or 'my_package.my_profile'
profile_name, cim = get_cim_profile()

line = cim.ACLineSegment(mRID='line-001', name='Feeder_Line_1')
terminal = cim.Terminal(mRID='term-001', name='Line_Terminal_1', ConductingEquipment=line)
line.Terminals.append(terminal)

print(f'Profile: {cim.__name__}')
print(f'Created {line.name} with mRID {line.mRID} and identifier {line.identifier}')
print(f'Created {terminal.name} with mRID {terminal.mRID} and identifier {terminal.identifier}')

Profile: cimprofiles.cim17.cimhub_2023
Created Feeder_Line_1 with mRID line-001 and identifier 62d1fade-3703-5c1e-ba4d-638ac7adcce9
Created Line_Terminal_1 with mRID term-001 and identifier c1b865e9-1472-57ff-b7fa-cda218b42327


## Common Issues and Troubleshooting

### Import Errors After Profile Generation

**Problem:** `ModuleNotFoundError: No module named 'my_package.my_profile'`

**Solutions:**
- Verify the profile directory contains both `my_profile.py` and `__init__.py`
- Ensure `__init__.py` imports from the profile's own module path (`my_package.my_profile.my_profile`), not `cimgraph.data_profile...`
- Check that the package is installed or on `sys.path` in the environment you run
- Restart the Python interpreter to clear the import cache

**Problem:** `ValueError: Unknown CIM profile 'my_profile': not an importable module path, ...`

**Solution:** `CIMG_CIM_PROFILE` is neither a known short name nor a module path whose top-level package can be imported. Use the full module path, e.g. `CIMG_CIM_PROFILE='my_package.my_profile'`, or register a short name in `cimprofiles.PROFILE_PACKAGES` (Option B).

### Invalid Dataclass Definitions

**Problem:** Generated dataclasses have syntax errors or missing attributes

**Solutions:**
- Check that the CIM UML model is properly loaded in CIMTool
- Ensure all required classes have proper inheritance chains
- Re-generate the profile with the latest builder versions

### Duplicate or Missing Associations

**Problem:** An association field appears twice, or is missing or incomplete

**Solutions:**
- Run `python -m cimprofiles.dedupe_profile_fields`, which removes duplicate fields and logs UML conflicts
- Check that both ends of the association are included in the profile
- Verify the inverse relationship is properly defined in CIMTool
- Ensure association cardinality is correctly specified

### Circular Import Issues

**Problem:** Circular import errors when using the profile

**Solutions:**
- Verify `from __future__ import annotations` is at the top of the generated file (the builder adds it)
- Check for custom modifications that may have broken imports

### Profile Doesn't Match XML Data

**Problem:** Cannot parse existing CIM XML with your profile

**Solutions:**
- Ensure the profile includes all classes present in the XML (the readers log classes that are not in the data profile)
- Check that the namespace URIs match between the profile (`cim.NAMESPACES`) and the XML
- Verify CIM version compatibility (CIM16, CIM17/CIM100, CIM18)

## Best Practices for Profile Development

### Start Small and Iterate
- Begin with the absolute minimum classes needed
- Test with sample data early and often
- Expand the profile gradually as requirements become clear
- Avoid "gold-plating" with unnecessary classes

### Maintain Profile Documentation
- Document the intended use case clearly
- Keep notes on why classes were included or excluded
- Track deviations from standard CIM patterns
- Maintain a changelog as the profile evolves

### Version Control Your Profiles
- Keep CIMTool `.owl` profile files in version control
- Tag stable versions (v1.0, v2.0, etc.)
- Document breaking changes between versions
- Consider semantic versioning

### Test Profile Compatibility
- Validate generated code imports correctly
- Test round-trip serialization (object → XML → object)
- Verify compatibility with target applications
- Check that all associations have proper inverses

### Coordinate Across Teams
- Share profiles with other teams using the same data
- Establish naming conventions for custom profiles
- Maintain a registry of available profiles
- Reuse existing profiles when possible instead of creating new ones

## Summary

Building custom CIM profiles with CIMTool enables you to:

1. **Reduce Complexity** - Work with only the CIM classes needed for your application
2. **Improve Performance** - Smaller profiles mean faster parsing and validation
3. **Enhance Maintainability** - Focused profiles are easier to understand and maintain
4. **Enable Type Safety** - Python dataclasses provide IDE support and type checking
5. **Facilitate Integration** - Custom profiles can bridge between different CIM versions or add extensions

### Key Takeaways:

- CIMTool Contextual Profiles define the UML subset
- XSLT builders transform profiles into usable formats
- CIMantic Graphs builders create Python dataclass schemas, which ship in the `cimprofiles` package
- Start small and expand profiles iteratively
- Test early and often with real data
- Maintain profile documentation and version control

### Next Steps:

- [Using Objects](2_3_using_objects.ipynb) - Work with CIM objects and their attributes
- [Using Incrementals](2_4_using_incrementals.ipynb) - Manage model changes using incrementals
- [Using Units](2_5_using_units.ipynb) - Handle physical quantities with proper units